# WildChat: English Multi-Turn Conversation Satisfaction Classification

**Scope:** English conversations with at least 2 user turns (multi-turn only)  
**Dataset:** [allenai/WildChat-1M](https://huggingface.co/datasets/allenai/WildChat-1M)  

## Pipeline Overview
1. **Data Loading** — Load the full WildChat dataset from HuggingFace
2. **Data Cleaning & Filtering** — English only, multi-turn (≥2 user turns), deduplication, content validation
3. **Exploratory Data Analysis** — Comprehensive statistics, distributions, and visualizations
4. **Feature Engineering** — Turn-level and conversation-level features for satisfaction prediction
5. **Weak Label Construction** — Heuristic-based satisfaction proxies from user messages
6. **Baseline Classification** — Logistic Regression + XGBoost with evaluation and interpretability

## 0 — Environment Setup

In [ ]:
import os, re, json, ast, math, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay
)
from sklearn.feature_extraction.text import HashingVectorizer

import xgboost as xgb

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

try:
    nltk.data.find("sentiment/vader_lexicon.zip")
except LookupError:
    nltk.download("vader_lexicon", quiet=True)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Environment ready.")

## 1 — Load Full Dataset

In [ ]:
from datasets import load_dataset

print("Loading WildChat-1M from HuggingFace (this may take a while)...")
hf_ds = load_dataset("allenai/WildChat-1M", split="train")
df_raw = hf_ds.to_pandas()
print(f"Loaded {len(df_raw):,} conversations.")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head(2)

## 2 — Data Cleaning & Filtering

Steps:
1. Normalize the nested conversation column into a consistent `list[dict]` format
2. Filter to **English** conversations only
3. Filter to **multi-turn** conversations (≥2 user turns)
4. Remove duplicates by `conversation_hash`
5. Validate conversation structure (proper user/assistant alternation)
6. Remove conversations with empty or extremely short content

In [ ]:
# --- 2a. Normalize conversation column ---

def normalize_conversation(conv: Any) -> List[Dict[str, Any]]:
    """Convert any storage format into list[dict] turns."""
    if isinstance(conv, str):
        s = conv.strip()
        if s and s[0] in "[{":
            try:
                conv = json.loads(s)
            except Exception:
                try:
                    conv = ast.literal_eval(s)
                except Exception:
                    return []
    if isinstance(conv, np.ndarray):
        conv = conv.tolist()
    if isinstance(conv, dict):
        keys = list(conv.keys())
        if not keys:
            return []
        lengths = [len(v) for v in conv.values() if isinstance(v, (list, tuple, np.ndarray))]
        n = max(lengths) if lengths else 0
        turns = []
        for j in range(n):
            t = {}
            for k in keys:
                v = conv[k]
                if isinstance(v, (list, tuple, np.ndarray)):
                    t[k] = v[j] if j < len(v) else None
                else:
                    t[k] = v
            turns.append(t)
        return turns
    if isinstance(conv, list):
        return [t for t in conv if isinstance(t, dict)]
    return []

print("Normalizing conversations...")
df_raw["_turns"] = [normalize_conversation(c) for c in tqdm(df_raw["conversation"].values, desc="Normalize")]
df_raw["_n_turns"] = df_raw["_turns"].apply(len)
df_raw["_n_user_turns"] = df_raw["_turns"].apply(lambda ts: sum(1 for t in ts if t.get("role") == "user"))
df_raw["_n_asst_turns"] = df_raw["_turns"].apply(lambda ts: sum(1 for t in ts if t.get("role") == "assistant"))

print(f"Total conversations: {len(df_raw):,}")
print(f"Turn count stats:\n{df_raw['_n_turns'].describe()}")

In [ ]:
# --- 2b. Filter to English only ---

# The dataset has a top-level 'language' column. Also check first user turn language.
def get_first_user_language(turns):
    for t in turns:
        if t.get("role") == "user":
            return t.get("language", None)
    return None

df_raw["_first_user_lang"] = df_raw["_turns"].apply(get_first_user_language)

# Use both the top-level language AND the per-turn language for robust filtering
is_english = (
    (df_raw["language"].str.lower() == "english") |
    (df_raw["_first_user_lang"].str.lower() == "english")
)

print(f"English conversations: {is_english.sum():,} / {len(df_raw):,} ({is_english.mean()*100:.1f}%)")

df_en = df_raw[is_english].copy()
print(f"After English filter: {len(df_en):,}")

In [ ]:
# --- 2c. Filter to multi-turn (>= 2 user turns) ---

is_multiturn = df_en["_n_user_turns"] >= 2
print(f"Multi-turn (≥2 user turns): {is_multiturn.sum():,} / {len(df_en):,} ({is_multiturn.mean()*100:.1f}%)")

df_mt = df_en[is_multiturn].copy()
print(f"After multi-turn filter: {len(df_mt):,}")

In [ ]:
# --- 2d. Deduplication ---

n_before = len(df_mt)
df_mt = df_mt.drop_duplicates(subset=["conversation_hash"], keep="first")
print(f"Deduplication: {n_before:,} -> {len(df_mt):,} (removed {n_before - len(df_mt):,} duplicates)")

In [ ]:
# --- 2e. Validate conversation structure ---

def is_valid_conversation(turns):
    """Check: non-empty, starts with user, has at least one assistant response."""
    if len(turns) < 2:
        return False
    if turns[0].get("role") != "user":
        return False
    roles = [t.get("role") for t in turns]
    if "assistant" not in roles:
        return False
    return True

valid_mask = df_mt["_turns"].apply(is_valid_conversation)
print(f"Valid structure: {valid_mask.sum():,} / {len(df_mt):,}")
df_mt = df_mt[valid_mask].copy()

In [ ]:
# --- 2f. Remove conversations with empty/very short content ---

def total_user_chars(turns):
    return sum(len(t.get("content", "") or "") for t in turns if t.get("role") == "user")

df_mt["_user_chars"] = df_mt["_turns"].apply(total_user_chars)

# Remove conversations where total user content is less than 20 characters (likely noise)
n_before = len(df_mt)
df_mt = df_mt[df_mt["_user_chars"] >= 20].copy()
print(f"Content length filter: {n_before:,} -> {len(df_mt):,} (removed {n_before - len(df_mt):,} too-short conversations)")

In [ ]:
# --- 2g. Cleaning summary ---

print("=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)
print(f"Raw dataset:                  {len(df_raw):>10,}")
print(f"After English filter:         {is_english.sum():>10,}")
print(f"After multi-turn filter:      {len(df_mt):>10,}")
print(f"Final cleaned dataset:        {len(df_mt):>10,}")
print(f"Retention rate:               {len(df_mt)/len(df_raw)*100:>9.1f}%")
print("=" * 60)

# Reset index for clean downstream processing
df = df_mt.reset_index(drop=True)
print(f"\nWorking dataset shape: {df.shape}")

## 3 — Exploratory Data Analysis

In [ ]:
# --- 3a. Basic statistics ---

print("Dataset Overview")
print(f"Conversations: {len(df):,}")
print(f"Total turns: {df['_n_turns'].sum():,}")
print(f"Total user turns: {df['_n_user_turns'].sum():,}")
print(f"Total assistant turns: {df['_n_asst_turns'].sum():,}")
print()

# Missing data
print("Missing data (%):\n")
# Exclude our temp columns from display
display_cols = [c for c in df.columns if not c.startswith("_")]
missing = (df[display_cols].isna().mean() * 100).round(2)
print(missing[missing > 0].to_string() if (missing > 0).any() else "No missing values in core columns.")
print()

# Data types
print("Column dtypes:")
print(df[display_cols].dtypes.to_string())

In [ ]:
# --- 3b. Turn count distribution ---

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df["_n_turns"].clip(upper=40), bins=40, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_title("Total Turns per Conversation")
axes[0].set_xlabel("Number of turns (clipped at 40)")
axes[0].set_ylabel("Count")
axes[0].axvline(df["_n_turns"].median(), color="red", linestyle="--", label=f"Median={df['_n_turns'].median():.0f}")
axes[0].legend()

axes[1].hist(df["_n_user_turns"].clip(upper=20), bins=20, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_title("User Turns per Conversation")
axes[1].set_xlabel("Number of user turns (clipped at 20)")
axes[1].set_ylabel("Count")
axes[1].axvline(df["_n_user_turns"].median(), color="red", linestyle="--", label=f"Median={df['_n_user_turns'].median():.0f}")
axes[1].legend()

axes[2].hist(df["_n_asst_turns"].clip(upper=20), bins=20, edgecolor="black", alpha=0.7, color="mediumseagreen")
axes[2].set_title("Assistant Turns per Conversation")
axes[2].set_xlabel("Number of assistant turns (clipped at 20)")
axes[2].set_ylabel("Count")
axes[2].axvline(df["_n_asst_turns"].median(), color="red", linestyle="--", label=f"Median={df['_n_asst_turns'].median():.0f}")
axes[2].legend()

plt.tight_layout()
plt.savefig("fig_turn_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTurn count statistics:")
df[["_n_turns", "_n_user_turns", "_n_asst_turns"]].describe().round(1)

In [ ]:
# --- 3c. Model distribution ---

model_counts = df["model"].value_counts()
print(f"Unique models: {model_counts.shape[0]}")
print()

fig, ax = plt.subplots(figsize=(10, 5))
top_n = 15
model_counts.head(top_n).plot(kind="barh", ax=ax, color="steelblue", edgecolor="black")
ax.set_title(f"Top {top_n} Models in English Multi-Turn Conversations")
ax.set_xlabel("Number of conversations")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("fig_model_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

model_counts.head(15)

In [ ]:
# --- 3d. Temporal distribution ---

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
df["date"] = df["timestamp"].dt.date

daily_counts = df.groupby("date").size()

fig, ax = plt.subplots(figsize=(14, 4))
daily_counts.plot(ax=ax, color="steelblue", alpha=0.7)
ax.set_title("Conversations Over Time (English Multi-Turn)")
ax.set_xlabel("Date")
ax.set_ylabel("Conversations per day")
plt.tight_layout()
plt.savefig("fig_temporal_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

In [ ]:
# --- 3e. Geographic distribution ---

country_counts = df["country"].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
country_counts.head(15).plot(kind="barh", ax=ax, color="coral", edgecolor="black")
ax.set_title("Top 15 Countries (English Multi-Turn)")
ax.set_xlabel("Number of conversations")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("fig_country_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

country_counts.head(15)

In [ ]:
# --- 3f. Toxicity & redaction prevalence ---

print("Toxicity and Redaction Stats")
print(f"  Toxic conversations:    {df['toxic'].sum():,} ({df['toxic'].mean()*100:.2f}%)")
print(f"  Redacted conversations: {df['redacted'].sum():,} ({df['redacted'].mean()*100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

toxic_counts = df["toxic"].value_counts()
axes[0].bar(["Non-toxic", "Toxic"], [toxic_counts.get(False, 0), toxic_counts.get(True, 0)],
            color=["steelblue", "tomato"], edgecolor="black")
axes[0].set_title("Toxicity Distribution")
axes[0].set_ylabel("Count")

redact_counts = df["redacted"].value_counts()
axes[1].bar(["Not redacted", "Redacted"], [redact_counts.get(False, 0), redact_counts.get(True, 0)],
            color=["steelblue", "orange"], edgecolor="black")
axes[1].set_title("Redaction Distribution")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("fig_toxicity_redaction.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 3g. Content length analysis ---

def compute_content_stats(turns):
    user_lens = [len(t.get("content", "") or "") for t in turns if t.get("role") == "user"]
    asst_lens = [len(t.get("content", "") or "") for t in turns if t.get("role") == "assistant"]
    return {
        "user_total_chars": sum(user_lens),
        "asst_total_chars": sum(asst_lens),
        "user_mean_chars": np.mean(user_lens) if user_lens else 0,
        "asst_mean_chars": np.mean(asst_lens) if asst_lens else 0,
        "user_max_chars": max(user_lens) if user_lens else 0,
        "asst_max_chars": max(asst_lens) if asst_lens else 0,
    }

content_stats = pd.DataFrame([compute_content_stats(t) for t in tqdm(df["_turns"].values, desc="Content stats")])
df = pd.concat([df.reset_index(drop=True), content_stats], axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["user_mean_chars"].clip(upper=2000), bins=50, alpha=0.7, color="coral", edgecolor="black", label="User")
axes[0].hist(df["asst_mean_chars"].clip(upper=2000), bins=50, alpha=0.5, color="steelblue", edgecolor="black", label="Assistant")
axes[0].set_title("Mean Message Length per Conversation")
axes[0].set_xlabel("Characters (clipped at 2000)")
axes[0].set_ylabel("Count")
axes[0].legend()

# Generation ratio: how much longer assistant messages are vs user
df["generation_ratio"] = df["asst_total_chars"] / df["user_total_chars"].clip(lower=1)
axes[1].hist(df["generation_ratio"].clip(upper=20), bins=50, alpha=0.7, color="mediumseagreen", edgecolor="black")
axes[1].set_title("Generation Ratio (Asst / User char length)")
axes[1].set_xlabel("Ratio (clipped at 20)")
axes[1].set_ylabel("Count")
axes[1].axvline(df["generation_ratio"].median(), color="red", linestyle="--",
                label=f"Median={df['generation_ratio'].median():.1f}")
axes[1].legend()

plt.tight_layout()
plt.savefig("fig_content_lengths.png", dpi=150, bbox_inches="tight")
plt.show()

content_stats.describe().round(1)

## 4 — Turn-Level Feature Engineering

In [ ]:
# --- 4a. Build turn-level DataFrame ---

def build_turn_df(df_conv, turns_col="_turns"):
    records = []
    conv_hashes = df_conv["conversation_hash"].values
    for i, turns in enumerate(tqdm(df_conv[turns_col].values, desc="Building turn-level DF")):
        cid = conv_hashes[i]
        for j, t in enumerate(turns):
            records.append({
                "conversation_id": cid,
                "turn_index": j,
                "role": t.get("role"),
                "content": t.get("content", "") or "",
            })
    return pd.DataFrame.from_records(records)

turn_df = build_turn_df(df)
print(f"Turn-level DataFrame: {len(turn_df):,} rows")
print(f"  User turns: {(turn_df['role'] == 'user').sum():,}")
print(f"  Assistant turns: {(turn_df['role'] == 'assistant').sum():,}")
turn_df.head(6)

In [ ]:
# --- 4b. Text features ---

turn_df["char_len"] = turn_df["content"].str.len()
turn_df["word_count"] = turn_df["content"].str.split().apply(len)
turn_df["has_question"] = turn_df["content"].str.contains(r"\?", regex=True).astype(int)
turn_df["n_questions"] = turn_df["content"].str.count(r"\?")
turn_df["n_exclamations"] = turn_df["content"].str.count(r"!")
turn_df["n_newlines"] = turn_df["content"].str.count(r"\n")
turn_df["n_code_blocks"] = turn_df["content"].str.count(r"```")
turn_df["has_url"] = turn_df["content"].str.contains(r"https?://", regex=True).astype(int)

# Refusal detection (assistant-side) — use non-capturing groups to avoid pandas warning
REFUSAL_PAT = re.compile(
    r"\b(?:i\s+can'?t|i\s+cannot|i\s+won'?t|i\s+am\s+unable|i'?m\s+not\s+able|"
    r"as\s+an?\s+ai|i\s+don'?t\s+have\s+the\s+ability|i\s+must\s+decline|i\s+apologize.*can'?t)\b",
    re.I
)
turn_df["has_refusal"] = turn_df["content"].str.contains(REFUSAL_PAT).astype(int)

print("Turn-level text features computed.")
turn_df[["role", "char_len", "word_count", "has_question", "has_refusal"]].groupby("role").describe().round(1)

In [ ]:
# --- 4c. Sentiment (VADER) ---

SIA = SentimentIntensityAnalyzer()

def vader_compound(text):
    if not text or not text.strip():
        return 0.0
    return SIA.polarity_scores(text)["compound"]

turn_df["sentiment"] = [vader_compound(t) for t in tqdm(turn_df["content"].values, desc="VADER sentiment")]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for i, role in enumerate(["user", "assistant"]):
    mask = turn_df["role"] == role
    axes[i].hist(turn_df.loc[mask, "sentiment"], bins=50, alpha=0.7,
                 color="coral" if role == "user" else "steelblue", edgecolor="black")
    axes[i].set_title(f"{role.title()} Sentiment Distribution")
    axes[i].set_xlabel("VADER Compound Score")
    axes[i].set_ylabel("Count")
    median_val = turn_df.loc[mask, "sentiment"].median()
    axes[i].axvline(median_val, color="red", linestyle="--", label=f"Median={median_val:.2f}")
    axes[i].legend()

plt.tight_layout()
plt.savefig("fig_sentiment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 4d. Topic similarity / coherence (HashingVectorizer) ---

HV = HashingVectorizer(
    n_features=2**18,
    alternate_sign=False,
    norm="l2",
    ngram_range=(1, 2),
    lowercase=True,
)

turn_df = turn_df.sort_values(["conversation_id", "turn_index"]).reset_index(drop=True)

print("Computing hashing vectors...")
X_hv = HV.transform(turn_df["content"].values)

# Adjacent cosine similarity (L2-normalized -> dot product = cosine)
adj_sim = np.array(X_hv[:-1].multiply(X_hv[1:]).sum(axis=1)).ravel()
turn_df["adjacent_sim"] = np.nan
turn_df.loc[1:, "adjacent_sim"] = adj_sim

# Null out cross-conversation boundaries
conv_changed = turn_df["conversation_id"].values[1:] != turn_df["conversation_id"].values[:-1]
turn_df.loc[np.where(conv_changed)[0] + 1, "adjacent_sim"] = np.nan

# User->Assistant alignment
turn_df["prev_role"] = turn_df.groupby("conversation_id")["role"].shift(1)
turn_df["ua_alignment"] = np.where(
    (turn_df["role"] == "assistant") & (turn_df["prev_role"] == "user"),
    turn_df["adjacent_sim"],
    np.nan
)

print(f"Adjacent similarity: mean={turn_df['adjacent_sim'].mean():.3f}, median={turn_df['adjacent_sim'].median():.3f}")
print(f"U->A alignment: mean={turn_df['ua_alignment'].mean():.3f}, median={turn_df['ua_alignment'].median():.3f}")

## 5 — Conversation-Level Feature Aggregation

In [ ]:
# --- 5a. Aggregate turn features to conversation level ---

def slope_of_series(y):
    y = np.asarray(y, dtype=float)
    y = y[~np.isnan(y)]
    if len(y) < 2:
        return np.nan
    x = np.arange(len(y), dtype=float)
    denom = np.sum((x - x.mean()) ** 2)
    if denom == 0:
        return np.nan
    return float(np.sum((x - x.mean()) * (y - y.mean())) / denom)

# Compute per-conversation sentiment slopes
print("Computing sentiment slopes...")
slopes = []
for cid, g in tqdm(turn_df.groupby("conversation_id", sort=False), desc="Sentiment slopes"):
    g = g.sort_values("turn_index")
    user_sent = g.loc[g["role"] == "user", "sentiment"].values
    asst_sent = g.loc[g["role"] == "assistant", "sentiment"].values
    slopes.append({
        "conversation_id": cid,
        "user_sent_slope": slope_of_series(user_sent),
        "asst_sent_slope": slope_of_series(asst_sent),
    })
slopes_df = pd.DataFrame(slopes)

# Main aggregation
print("Aggregating conversation-level features...")
agg = turn_df.groupby("conversation_id").agg(
    n_turns=("turn_index", "count"),
    mean_char_len=("char_len", "mean"),
    std_char_len=("char_len", "std"),
    mean_word_count=("word_count", "mean"),
    mean_sentiment=("sentiment", "mean"),
    std_sentiment=("sentiment", "std"),
    min_sentiment=("sentiment", "min"),
    max_sentiment=("sentiment", "max"),
    mean_adjacent_sim=("adjacent_sim", "mean"),
    mean_ua_alignment=("ua_alignment", "mean"),
    frac_refusal=("has_refusal", "mean"),
    frac_questions=("has_question", "mean"),
    total_questions=("n_questions", "sum"),
    total_code_blocks=("n_code_blocks", "sum"),
).reset_index()

# User-only and assistant-only aggregations
user_agg = turn_df[turn_df["role"] == "user"].groupby("conversation_id").agg(
    user_mean_sentiment=("sentiment", "mean"),
    user_mean_char_len=("char_len", "mean"),
    user_mean_word_count=("word_count", "mean"),
    user_total_questions=("n_questions", "sum"),
    user_frac_questions=("has_question", "mean"),
).reset_index()

asst_agg = turn_df[turn_df["role"] == "assistant"].groupby("conversation_id").agg(
    asst_mean_sentiment=("sentiment", "mean"),
    asst_mean_char_len=("char_len", "mean"),
    asst_mean_word_count=("word_count", "mean"),
    asst_frac_refusal=("has_refusal", "mean"),
    asst_total_code_blocks=("n_code_blocks", "sum"),
).reset_index()

# Last user turn features (important for satisfaction signal)
last_user = turn_df[turn_df["role"] == "user"].sort_values(
    ["conversation_id", "turn_index"]
).groupby("conversation_id").tail(1)[["conversation_id", "sentiment", "char_len", "has_question"]].rename(
    columns={"sentiment": "last_user_sentiment", "char_len": "last_user_char_len", "has_question": "last_user_has_question"}
)

# Merge everything
conv_features = (
    agg
    .merge(slopes_df, on="conversation_id", how="left")
    .merge(user_agg, on="conversation_id", how="left")
    .merge(asst_agg, on="conversation_id", how="left")
    .merge(last_user, on="conversation_id", how="left")
)

# Add conversation-level metadata
meta_cols = ["conversation_hash", "model", "toxic", "redacted"]
conv_features = conv_features.merge(
    df[meta_cols].rename(columns={"conversation_hash": "conversation_id"}),
    on="conversation_id", how="left"
)
conv_features["toxic"] = conv_features["toxic"].astype(int)
conv_features["redacted"] = conv_features["redacted"].astype(int)

print(f"Conversation features shape: {conv_features.shape}")
conv_features.head(3)

In [ ]:
# --- 5b. Feature correlation heatmap ---

numeric_cols = conv_features.select_dtypes(include=[np.number]).columns.tolist()
# Remove ID-like columns
numeric_cols = [c for c in numeric_cols if c != "conversation_id"]

corr = conv_features[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, annot=False,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("fig_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 6 — Weak Satisfaction Labels

Since WildChat does not include explicit user satisfaction labels, we construct **weak/proxy labels** from textual cues in user messages.

### Labeling Strategy
We scan user messages (with emphasis on **later turns** which carry more signal) for:

- **Positive signals**: explicit gratitude, confirmation, praise
- **Negative signals**: complaints, corrections, expressions of frustration
- **Behavioral signals**: user abandonment patterns, repeated questions

In [ ]:
# --- 6a. Enhanced weak labeling ---

POS_PATTERNS = [
    re.compile(r"\b(thanks|thank\s+you|thx|ty)\b", re.I),
    re.compile(r"\b(appreciate\s+(it|that|this|your)|that\s+helps?|very\s+helpful|so\s+helpful)\b", re.I),
    re.compile(r"\b(perfect|excellent|awesome|amazing|wonderful|fantastic|brilliant|great\s+(job|work|answer|response|explanation))\b", re.I),
    re.compile(r"\b(exactly\s+what\s+i|that'?s\s+(exactly|precisely)|just\s+what\s+i\s+needed)\b", re.I),
    re.compile(r"\b(works?\s+(perfectly|great|well|fine)|that\s+worked|it\s+works|solved\s+(it|my))\b", re.I),
    re.compile(r"\b(well\s+done|good\s+(job|work|answer)|nice\s+(job|work)|nailed\s+it)\b", re.I),
    re.compile(r"\b(you'?re\s+(the\s+best|amazing|awesome|great|helpful))\b", re.I),
]

NEG_PATTERNS = [
    re.compile(r"\b(not\s+helpful|unhelpful|useless|pointless|waste\s+of\s+time)\b", re.I),
    re.compile(r"\b(wrong|incorrect|inaccurate|false|that'?s\s+not\s+(right|correct|true)|you'?re\s+wrong)\b", re.I),
    re.compile(r"\b(doesn'?t\s+work|does\s+not\s+work|didn'?t\s+work|still\s+(not|doesn'?t|won'?t)\s+work)\b", re.I),
    re.compile(r"\b(that'?s\s+not\s+what\s+i\s+(asked|meant|wanted)|you\s+(didn'?t|don'?t)\s+(understand|answer|address))\b", re.I),
    re.compile(r"\b(terrible|horrible|awful|pathetic|incompetent|stupid\s+answer)\b", re.I),
    re.compile(r"\b(try\s+again|let\s+me\s+rephrase|i'?ll\s+ask\s+again|please\s+(re-?read|actually\s+read))\b", re.I),
    re.compile(r"\b(you\s+keep\s+(saying|repeating|giving)|you\s+already\s+said\s+that|same\s+(thing|answer)\s+again)\b", re.I),
    re.compile(r"\b(no[,.]?\s+that'?s\s+not|no[,.]?\s+i\s+(said|meant|asked|wanted))\b", re.I),
]

def count_pattern_matches(text, patterns):
    return sum(1 for p in patterns if p.search(text))

def weak_satisfaction_label(turns):
    """Enhanced weak labeling with weighted scoring.
    Later turns get higher weight since they better reflect final satisfaction.
    Returns: 1 (satisfied), 0 (unsatisfied), None (unlabeled)
    """
    user_texts = []
    for t in turns:
        if t.get("role") == "user":
            content = t.get("content", "") or ""
            user_texts.append(content)

    if not user_texts:
        return None

    # Weight later turns more heavily (last turn 2x, second-to-last 1.5x)
    n = len(user_texts)
    pos_score = 0.0
    neg_score = 0.0

    for i, txt in enumerate(user_texts):
        if not txt.strip():
            continue
        weight = 1.0
        if i == n - 1:
            weight = 2.0  # last user message
        elif i == n - 2:
            weight = 1.5  # second-to-last

        pos_score += count_pattern_matches(txt, POS_PATTERNS) * weight
        neg_score += count_pattern_matches(txt, NEG_PATTERNS) * weight

    if pos_score > 0 and neg_score == 0:
        return 1
    if neg_score > 0 and pos_score == 0:
        return 0
    if pos_score > 0 and neg_score > 0:
        # Mixed signals: use the dominant one with a margin
        if pos_score >= neg_score * 2:
            return 1
        if neg_score >= pos_score * 2:
            return 0
        return None  # too ambiguous
    return None  # no signal


print("Computing weak satisfaction labels...")
conv_id_to_turns = dict(zip(df["conversation_hash"], df["_turns"]))

labels = []
for cid in tqdm(conv_features["conversation_id"].values, desc="Weak labels"):
    turns = conv_id_to_turns.get(cid, [])
    labels.append(weak_satisfaction_label(turns))

conv_features["satisfaction"] = labels

label_dist = conv_features["satisfaction"].value_counts(dropna=False)
print("\nLabel distribution:")
print(label_dist)
print(f"\nLabeled: {conv_features['satisfaction'].notna().sum():,} ({conv_features['satisfaction'].notna().mean()*100:.1f}%)")
print(f"Unlabeled: {conv_features['satisfaction'].isna().sum():,} ({conv_features['satisfaction'].isna().mean()*100:.1f}%)")

In [ ]:
# --- 6b. Label distribution visualization ---

labeled_df = conv_features.dropna(subset=["satisfaction"]).copy()
labeled_df["satisfaction"] = labeled_df["satisfaction"].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall distribution
sat_counts = labeled_df["satisfaction"].value_counts().sort_index()
axes[0].bar(["Unsatisfied (0)", "Satisfied (1)"], sat_counts.values,
            color=["tomato", "mediumseagreen"], edgecolor="black")
axes[0].set_title(f"Weak Label Distribution (n={len(labeled_df):,})")
axes[0].set_ylabel("Count")
for i, v in enumerate(sat_counts.values):
    axes[0].text(i, v + len(labeled_df) * 0.01, f"{v:,}\n({v/len(labeled_df)*100:.1f}%)",
                ha="center", fontsize=10)

# Pie chart
total = len(conv_features)
labeled_n = labeled_df.shape[0]
unlabeled_n = total - labeled_n
axes[1].pie([sat_counts.get(1, 0), sat_counts.get(0, 0), unlabeled_n],
            labels=["Satisfied", "Unsatisfied", "Unlabeled"],
            colors=["mediumseagreen", "tomato", "lightgray"],
            autopct="%1.1f%%", startangle=90)
axes[1].set_title(f"All Conversations (n={total:,})")

plt.tight_layout()
plt.savefig("fig_label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 6c. Compare feature distributions by satisfaction label ---

key_features = [
    "n_turns", "mean_sentiment", "user_mean_sentiment", "last_user_sentiment",
    "mean_ua_alignment", "asst_frac_refusal", "user_sent_slope",
    "generation_ratio" if "generation_ratio" in labeled_df.columns else "mean_char_len",
]
# Filter to features actually present
key_features = [f for f in key_features if f in labeled_df.columns]

n_feat = len(key_features)
n_cols = 4
n_rows = math.ceil(n_feat / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if n_feat == 1 else axes.flatten()

for i, feat in enumerate(key_features):
    ax = axes[i]
    for label, color, name in [(1, "mediumseagreen", "Satisfied"), (0, "tomato", "Unsatisfied")]:
        data = labeled_df.loc[labeled_df["satisfaction"] == label, feat].dropna()
        ax.hist(data.clip(lower=data.quantile(0.01), upper=data.quantile(0.99)),
                bins=30, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions by Satisfaction Label", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("fig_feature_distributions_by_label.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 — Baseline Satisfaction Classifier

In [ ]:
# --- 7a. Prepare data ---

FEATURE_COLS = [
    "n_turns",
    "mean_char_len", "std_char_len",
    "mean_word_count",
    "mean_sentiment", "std_sentiment", "min_sentiment", "max_sentiment",
    "mean_adjacent_sim", "mean_ua_alignment",
    "frac_refusal", "frac_questions", "total_questions",
    "total_code_blocks",
    "user_mean_sentiment", "user_mean_char_len", "user_mean_word_count",
    "user_total_questions", "user_frac_questions",
    "asst_mean_sentiment", "asst_mean_char_len", "asst_mean_word_count",
    "asst_frac_refusal", "asst_total_code_blocks",
    "last_user_sentiment", "last_user_char_len", "last_user_has_question",
    "user_sent_slope", "asst_sent_slope",
    "toxic", "redacted",
]

# Filter to available columns
FEATURE_COLS = [c for c in FEATURE_COLS if c in conv_features.columns]
print(f"Using {len(FEATURE_COLS)} features: {FEATURE_COLS}")

labeled = conv_features.dropna(subset=["satisfaction"]).copy()
y = labeled["satisfaction"].astype(int)
X = labeled[FEATURE_COLS].copy()

# Convert bool columns to int
for c in X.columns:
    if X[c].dtype == bool:
        X[c] = X[c].astype(int)

print(f"\nLabeled samples: {len(X):,}")
print(f"Class distribution: {y.value_counts().to_dict()}")
print(f"Class balance: {y.mean()*100:.1f}% positive")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
print(f"\nTrain: {len(X_train):,}  |  Test: {len(X_test):,}")

In [ ]:
# --- 7b. Logistic Regression baseline ---

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

lr_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_SEED)),
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

print("=" * 60)
print("LOGISTIC REGRESSION")
print("=" * 60)
print(classification_report(y_test, y_pred_lr, digits=3, target_names=["Unsatisfied", "Satisfied"]))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob_lr):.3f}")
print(f"Average Precision: {average_precision_score(y_test, y_prob_lr):.3f}")

In [ ]:
# --- 7c. XGBoost classifier ---

# Compute scale_pos_weight for imbalanced classes
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

xgb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", xgb.XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_SEED,
        eval_metric="logloss",
    )),
])

xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = xgb_pipeline.predict(X_test)
y_prob_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]

print("=" * 60)
print("XGBOOST")
print("=" * 60)
print(classification_report(y_test, y_pred_xgb, digits=3, target_names=["Unsatisfied", "Satisfied"]))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob_xgb):.3f}")
print(f"Average Precision: {average_precision_score(y_test, y_prob_xgb):.3f}")

In [ ]:
# --- 7d. Cross-validation ---

print("5-Fold Cross-Validation (ROC AUC):")

X_full_imputed = SimpleImputer(strategy="median").fit_transform(X)

# Logistic Regression CV
lr_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_SEED)),
])
lr_scores = cross_val_score(lr_cv, X_full_imputed, y, cv=5, scoring="roc_auc")
print(f"  Logistic Regression: {lr_scores.mean():.3f} +/- {lr_scores.std():.3f}")

# XGBoost CV
xgb_cv = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, random_state=RANDOM_SEED,
    eval_metric="logloss",
)
xgb_scores = cross_val_score(xgb_cv, X_full_imputed, y, cv=5, scoring="roc_auc")
print(f"  XGBoost:             {xgb_scores.mean():.3f} +/- {xgb_scores.std():.3f}")

In [ ]:
# --- 7e. Confusion matrices ---

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, title in [
    (axes[0], y_pred_lr, "Logistic Regression"),
    (axes[1], y_pred_xgb, "XGBoost"),
]:
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Unsatisfied", "Satisfied"]).plot(ax=ax, cmap="Blues")
    ax.set_title(title)

plt.tight_layout()
plt.savefig("fig_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 8 — Feature Importance & Interpretability

In [ ]:
# --- 8a. Logistic Regression coefficients ---

lr_model = lr_pipeline.named_steps["clf"]
lr_coefs = pd.Series(lr_model.coef_[0], index=FEATURE_COLS).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["tomato" if v < 0 else "mediumseagreen" for v in lr_coefs.values]
lr_coefs.plot(kind="barh", ax=ax, color=colors, edgecolor="black")
ax.set_title("Logistic Regression Coefficients\n(positive = more likely satisfied)")
ax.set_xlabel("Coefficient")
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig("fig_lr_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top 5 positive predictors:")
print(lr_coefs.tail(5).to_string())
print("\nTop 5 negative predictors:")
print(lr_coefs.head(5).to_string())

In [ ]:
# --- 8b. XGBoost feature importance ---

xgb_model = xgb_pipeline.named_steps["clf"]
xgb_importance = pd.Series(
    xgb_model.feature_importances_, index=FEATURE_COLS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
xgb_importance.plot(kind="barh", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("XGBoost Feature Importance (Gain)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("fig_xgb_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top 10 most important features:")
print(xgb_importance.tail(10).to_string())

## 9 — Export & Summary

In [ ]:
# --- 9a. Export features ---

os.makedirs("outputs", exist_ok=True)

conv_features.to_parquet("outputs/wildchat_en_multiturn_conv_features.parquet", index=False)
turn_df.to_parquet("outputs/wildchat_en_multiturn_turn_features.parquet", index=False)

print("Exported:")
print(f"  Conversation features: outputs/wildchat_en_multiturn_conv_features.parquet ({len(conv_features):,} rows)")
print(f"  Turn features: outputs/wildchat_en_multiturn_turn_features.parquet ({len(turn_df):,} rows)")

In [ ]:
# --- 9b. Final summary ---

print("=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)
print(f"")
print(f"Dataset:       WildChat-1M (English, multi-turn only)")
print(f"Raw size:      {len(df_raw):>10,} conversations")
print(f"Cleaned size:  {len(df):>10,} conversations")
print(f"Turn-level:    {len(turn_df):>10,} turns")
print(f"")
print(f"Features:      {len(FEATURE_COLS)} conversation-level features")
print(f"Labeled:       {conv_features['satisfaction'].notna().sum():>10,} conversations")
print(f"  Satisfied:   {(conv_features['satisfaction'] == 1).sum():>10,}")
print(f"  Unsatisfied: {(conv_features['satisfaction'] == 0).sum():>10,}")
print(f"")
print(f"Baseline Results (Test Set):")
print(f"  Logistic Regression ROC AUC: {roc_auc_score(y_test, y_prob_lr):.3f}")
print(f"  XGBoost ROC AUC:             {roc_auc_score(y_test, y_prob_xgb):.3f}")
print(f"")
print(f"Cross-Validation ROC AUC:")
print(f"  Logistic Regression: {lr_scores.mean():.3f} +/- {lr_scores.std():.3f}")
print(f"  XGBoost:             {xgb_scores.mean():.3f} +/- {xgb_scores.std():.3f}")
print(f"")
print("=" * 70)
print("Next steps:")
print("  1. Manually label 500-2000 conversations for gold-standard evaluation")
print("  2. Add embedding-based features (sentence-transformers)")
print("  3. Try text-based classifiers (fine-tuned BERT on last user turn)")
print("  4. Improve weak labels with LLM-based annotation")
print("  5. Add conversation-flow features (topic switching, repetition)")
print("=" * 70)